# task_background_color_change_01

このノートブックは `annotations_gt_task_ver10.json` から `tasks[0].action == replace_background` の先頭エントリを探し、`replace_background` を単体実行して確認するための検証用です。

前景を検出して背景を置き換える（ぼかしまたは色で塗る）

In [ ]:
from pathlib import Path
import json
import sys
import logging

WORKSPACE = Path('/workspace')
VIDEO_DIR = WORKSPACE / 'data' / 'videos'
RULES_PATH = WORKSPACE / 'logs' / 'submit' / 'submission_ver05_json' / 'task_rules_ver05.json'
TASK_DECOMP_PATH = WORKSPACE / 'logs' / 'submit' / 'submission_ver05_json' / 'task_decomposition_ver05.json'
ANNOT_PATH = WORKSPACE / 'data' / 'annotations_gt_task_ver10.json'
OUT_DIR = WORKSPACE / 'logs' / 'notebook' / 'task_background_color_change_01'
OUT_DIR.mkdir(parents=True, exist_ok=True)

if str(WORKSPACE) not in sys.path:
    sys.path.insert(0, str(WORKSPACE))

from backup import task_rules_ver05_functions as task_rule_funcs
from src.utils.video_utility import load_video, write_video, show_before_after

rules_payload = json.loads(RULES_PATH.read_text(encoding='utf-8'))
rules = rules_payload['actions']
task_rows = json.loads(TASK_DECOMP_PATH.read_text(encoding='utf-8'))
instruction_by_video = {str(r.get('video_path', '')): str(r.get('instruction', '')) for r in task_rows}

logger = logging.getLogger('task_background_color_change_01')
logger.setLevel(logging.INFO)
if not logger.handlers:
    h = logging.StreamHandler()
    h.setLevel(logging.INFO)
    h.setFormatter(logging.Formatter('[%(levelname)s] %(message)s'))
    logger.addHandler(h)

def run_action_core(video_path_in, video_path_out, action, params_override=None, instruction_override=None, show_compare=False):
    frames, fps, width, height = load_video(video_path_in)
    rule = dict(rules.get(action, {}))
    method = str(rule.get('method', 'identity'))

    params = dict(rule.get('params', {}))
    if params_override:
        params.update(params_override)

    if instruction_override is None:
        instruction = instruction_by_video.get(Path(video_path_in).name, '')
    else:
        instruction = instruction_override

    out_frames = task_rule_funcs.run_method(
        method=method,
        frames=frames,
        params=params,
        instruction=instruction,
        logger=logger,
    )
    write_video(video_path_out, out_frames, fps, width, height)
    if show_compare:
        show_before_after(video_path_in, video_path_out)
    return instruction, method, params

print('rules :', len(rules))
print('RULES_PATH:', RULES_PATH)
print('TASK_DECOMP_PATH:', TASK_DECOMP_PATH)
print('ANNOT_PATH:', ANNOT_PATH)

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


rules : 44
RULES_PATH: /workspace/logs/submit/submission_ver05_json/task_rules_ver05.json
TASK_DECOMP_PATH: /workspace/logs/submit/submission_ver05_json/task_decomposition_ver05.json
ANNOT_PATH: /workspace/data/annotations_gt_task_ver10.json


In [2]:
# tasks[0].action == replace_background の先頭エントリを準備
SHOW_COMPARE = True
BLUR_BACKGROUND = True  # Trueの場合は背景をぼかす、Falseの場合は色で塗る

ann_rows = json.loads(ANNOT_PATH.read_text(encoding='utf-8'))
if not ann_rows:
    raise ValueError('annotations is empty')

ANNOTATION_ID = None
entry = None
for i, row in enumerate(ann_rows):
    tasks = row.get('tasks', [])
    if tasks and str(tasks[0].get('action', '')) == 'replace_background':
        ANNOTATION_ID = i
        entry = dict(row)
        break

if entry is None:
    raise ValueError('tasks[0].action == replace_background のエントリが見つかりません')

TARGET_VIDEO = str(entry.get('video_path', ''))
if not TARGET_VIDEO:
    raise ValueError('selected annotation has empty video_path')

first_task = dict(entry.get('tasks', [])[0])
instruction = str(entry.get('instruction', ''))
video_in = VIDEO_DIR / TARGET_VIDEO
if not video_in.exists():
    raise FileNotFoundError(f'input video not found: {video_in}')

SEQ_OUT_DIR = OUT_DIR / f'replace_background_top{ANNOTATION_ID+1}' / Path(TARGET_VIDEO).stem
SEQ_OUT_DIR.mkdir(parents=True, exist_ok=True)

print('annotation id  :', ANNOTATION_ID)
print('target video   :', TARGET_VIDEO)
print('class/subclass :', entry.get('class'), '/', entry.get('subclass'))
print('instruction    :', instruction)
print('gt task action :', first_task.get('action', ''))
print('blur background:', BLUR_BACKGROUND)

annotation id  : 3
target video   : DaUJkmBvTKM_2_0to150.mp4
class/subclass : Visual Effect Editing / Background Change
instruction    : Replace the entire outdoor background behind the speaker with a sleek, modern automotive showroom featuring soft studio lighting and blurred cars in the distance. Ensure that the speaker's outline, including the fine edges of his head and shoulders, is cleanly masked with no edge flickering or artifacts across all frames. The existing lower-third text and the 'an' logo in the top right must remain perfectly legible and completely unaffected by the background modification. Maintain a high-quality, professional look where the new background's lighting and perspective naturally complement the speaker's position in the frame.
gt task action : replace_background
blur background: True


In [3]:
# replace_background を実行
import importlib
importlib.reload(task_rule_funcs)

action = 'replace_background'
params = dict(first_task.get('params', {}))
params['_action'] = action
params['_constraints'] = list(first_task.get('constraints', []))
params['action'] = action
params['constraints'] = list(first_task.get('constraints', []))
params['blur_background'] = BLUR_BACKGROUND
params['video_path'] = TARGET_VIDEO
if first_task.get('target', None) is not None:
    params['target'] = first_task.get('target', '')

out_path = SEQ_OUT_DIR / f"{Path(TARGET_VIDEO).stem}__task01_{action}{'_blur' if BLUR_BACKGROUND else '_color'}.mp4"
_instruction, method, used_params = run_action_core(
    video_in,
    out_path,
    action,
    params_override=params,
    instruction_override=instruction,
    show_compare=SHOW_COMPARE,
)

print('video      :', video_in.name)
print('action     :', action)
print('method     :', method)
print('out        :', out_path)
print('used params:', used_params)

KeyboardInterrupt: 

In [ ]:
# 結果の視覚化と検証
import cv2
import numpy as np
import matplotlib.pyplot as plt

# 出力動画のフレームを読み込む
frames_out = []
cap_out = cv2.VideoCapture(str(out_path))
while True:
    ret, frame = cap_out.read()
    if not ret:
        break
    frames_out.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
cap_out.release()

# 入力動画のフレームを読み込む
frames_in = []
cap_in = cv2.VideoCapture(str(video_in))
while True:
    ret, frame = cap_in.read()
    if not ret:
        break
    frames_in.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
cap_in.release()

print(f'入力フレーム数: {len(frames_in)}')
print(f'出力フレーム数: {len(frames_out)}')
print(f'フレームサイズ: {frames_in[0].shape}')
print(f'出力パス: {out_path}')

In [ ]:
# Before/After の比較グリッドを表示
# 均等に分散した複数フレームを表示
frame_indices = np.linspace(0, len(frames_in) - 1, min(10, len(frames_in)), dtype=int)

cols = 5
rows = 4  # Before 2行 + After 2行
fig, axes = plt.subplots(rows, cols, figsize=(20, 12))

# Before フレーム（最初の2行）
for col_idx, frame_idx in enumerate(frame_indices[:cols]):
    # 上の行：入力フレーム
    ax = axes[0, col_idx]
    ax.imshow(frames_in[frame_idx])
    ax.set_title(f'Frame {frame_idx} (Input)')
    ax.axis('off')
    
    # 下の行：出力フレーム
    ax = axes[1, col_idx]
    if frame_idx < len(frames_out):
        ax.imshow(frames_out[frame_idx])
        ax.set_title(f'Frame {frame_idx} (Output)')
    ax.axis('off')

# After フレーム（最後の2行）
for col_idx, frame_idx in enumerate(frame_indices[cols:]):
    # 上の行：入力フレーム
    ax = axes[2, col_idx]
    ax.imshow(frames_in[frame_idx])
    ax.set_title(f'Frame {frame_idx} (Input)')
    ax.axis('off')
    
    # 下の行：出力フレーム
    ax = axes[3, col_idx]
    if frame_idx < len(frames_out):
        ax.imshow(frames_out[frame_idx])
        ax.set_title(f'Frame {frame_idx} (Output)')
    ax.axis('off')

plt.tight_layout()
plt.savefig(str(SEQ_OUT_DIR / 'comparison_grid.png'), dpi=100)
plt.show()

print('Comparison grid saved')

In [ ]:
# 出力動画を表示（Jupyter内）
from IPython.display import Video, HTML

# 最初のフレームを表示
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

ax1.imshow(frames_in[0])
ax1.set_title('Original Frame (First)')
ax1.axis('off')

ax2.imshow(frames_out[0])
ax2.set_title('Processed Frame (First)')
ax2.axis('off')

plt.tight_layout()
plt.show()

# 出力動画ファイルの情報
if out_path.exists():
    print(f'✓ Output video created: {out_path}')
    print(f'  File size: {out_path.stat().st_size / (1024*1024):.2f} MB')
    
    # 動画を埋め込み表示
    try:
        display(Video(str(out_path), embed=True, width=640, height=360))
    except Exception as e:
        print(f'Could not embed video: {e}')
else:
    print('✗ Output video not found')

In [ ]:
# 処理の詳細ログを表示
print('=' * 80)
print('replace_background EXECUTION SUMMARY')
print('=' * 80)
print()
print('📹 VIDEO INFORMATION')
print(f'  Input video: {video_in.name}')
print(f'  Output video: {out_path.name}')
print(f'  Output directory: {SEQ_OUT_DIR}')
print()
print('📋 ANNOTATION DETAILS')
print(f'  Annotation ID: {ANNOTATION_ID}')
print(f'  Class: {entry.get("class")}')
print(f'  Subclass: {entry.get("subclass")}')
print(f'  Target (from task): {first_task.get("target", "N/A")}')
print()
print('🎯 INSTRUCTION')
print(f'  {instruction}')
print()
print('⚙️  PROCESSING PARAMETERS')
print(f'  Action: {action}')
print(f'  Method: {method}')
print(f'  Blur background: {BLUR_BACKGROUND}')
for key, value in used_params.items():
    if key not in ['video_path', 'action', 'constraints', '_action', '_constraints']:
        print(f'  {key}: {value}')
print()
print('📊 FRAME STATISTICS')
print(f'  Total frames: {len(frames_in)}')
print(f'  Frame resolution: {frames_in[0].shape[1]}x{frames_in[0].shape[0]}')
print(f'  Output frames: {len(frames_out)}')
print()
print('✅ Processing complete!')
print('=' * 80)

In [ ]:
# オプション: 別の手法を試す（背景を色で塗る場合）
# BLUR_BACKGROUND = False に変更して再実行

print("=" * 80)
print("ALTERNATIVE: Replace background with solid color")
print("=" * 80)
print()
print("To try filling the background with a solid color instead of blurring,")
print("execute the following cell after changing BLUR_BACKGROUND to False.")
print()
print("Changes required:")
print("1. Set BLUR_BACKGROUND = False in the parameter cell")
print("2. Re-run the replace_background execution cell")
print()
print("The instruction text will be parsed to determine the target color.")
print("=" * 80)